# 코트 키포인트 모델 기준선 학습 (Colab)

11/11 학습과 **같은 데이터·설정·버전**으로 다시 학습해 자동 보정의 기준선을 만든다.

준비 (한 번만)
1. 로컬 `calibration/train/court_pose_dataset.zip` (717 MB, 1103 경기 2,351장)을 Google Drive `내 드라이브/court_pose/`에 올린다.
2. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**.
3. **런타임 → 모두 실행**. 연결이 끊기면 다시 모두 실행하면 마지막 체크포인트에서 이어서 학습한다.

끝나면 `내 드라이브/court_pose/baseline/best.pt`를 받아 로컬 `D:\02_Projects\futsal\calibration\train\best.pt`에 넣는다.

In [ ]:
# 1. GPU 확인 (없으면 여기서 멈춘다: CPU로는 며칠 걸린다)
import torch, subprocess
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경 > T4 GPU 로 바꾼 뒤 다시 실행하세요'

In [ ]:
# 2. Drive 연결, 경로
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/court_pose'
ZIP = os.path.join(DRIVE, 'court_pose_dataset.zip')
RUNS = os.path.join(DRIVE, 'runs')
NAME = 'baseline'                      # 결과: {RUNS}/baseline/weights/{last,best}.pt
OUT = os.path.join(DRIVE, 'baseline')  # 최종 best.pt와 지표를 모아 두는 곳
assert os.path.exists(ZIP), f'데이터가 없습니다: {ZIP} (로컬 calibration/train/court_pose_dataset.zip을 올리세요)'
os.makedirs(OUT, exist_ok=True)
print('데이터:', ZIP, f'{os.path.getsize(ZIP) / 1e6:.0f} MB')

In [ ]:
# 3. 로컬 프로젝트(11/11 학습)와 같은 ultralytics 버전, 데이터 압축 해제 (Colab 로컬 디스크로: Drive에서 직접 읽으면 느리다)
!pip install -q ultralytics==8.4.183
import zipfile
DATA_DIR = '/content/court_pose'
if not os.path.exists(os.path.join(DATA_DIR, 'data.yaml')):
    zipfile.ZipFile(ZIP).extractall(DATA_DIR)
DATA = os.path.join(DATA_DIR, 'data.yaml')
print(open(DATA).read())
print('학습', len(os.listdir(DATA_DIR + '/images/train')), '/ 검증', len(os.listdir(DATA_DIR + '/images/val')))

In [ ]:
# 4. 학습: 11/11과 같은 설정. 체크포인트(last.pt)가 Drive에 있으면 이어서 학습한다.
from ultralytics import YOLO
last = os.path.join(RUNS, NAME, 'weights', 'last.pt')
if os.path.exists(last):
    print('이어서 학습:', last)
    YOLO(last).train(resume=True)
else:
    YOLO('yolo11n-pose.pt').train(
        data=DATA, imgsz=960, epochs=100, batch=16, patience=25, seed=0, deterministic=True,
        mosaic=0.0, degrees=0.0, shear=0.0, perspective=0.0003,
        translate=0.1, scale=0.3, fliplr=0.0,          # 좌우 반전 끔: 반전 영상은 실제로 나오지 않는다
        hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
        project=RUNS, name=NAME, exist_ok=True, save_period=10,
    )

In [ ]:
# 5. 검증 지표를 재고 best.pt와 함께 Drive에 모아 둔다
import json, shutil, ultralytics, datetime
best_path = os.path.join(RUNS, NAME, 'weights', 'best.pt')
m = YOLO(best_path).val(data=DATA, imgsz=960)
metrics = {
    'time': datetime.datetime.now().isoformat(timespec='seconds'),
    'ultralytics': ultralytics.__version__, 'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0),
    'pose_map50': float(m.pose.map50), 'pose_map50_95': float(m.pose.map),
    'box_map50': float(m.box.map50),
    'data': 'court_pose_dataset.zip (1103, 2026-11-11, 585711f)',
    'settings': 'yolo11n-pose imgsz=960 epochs=100 batch=16 patience=25 fliplr=0 mosaic=0',
}
shutil.copy(best_path, os.path.join(OUT, 'best.pt'))
for f in ('results.csv', 'args.yaml'):
    src = os.path.join(RUNS, NAME, f)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(OUT, f))
json.dump(metrics, open(os.path.join(OUT, 'metrics.json'), 'w'), indent=1)
print(json.dumps(metrics, indent=1, ensure_ascii=False))
print('\n받을 파일:', os.path.join(OUT, 'best.pt'))

In [ ]:
# 6. (선택) 바로 내려받기 - Drive에서 받아도 된다
from google.colab import files
files.download(os.path.join(OUT, 'best.pt'))